# **PROJETO TMBA - Google Play Store App**

### **Projeto desenvolvido por:**
- Débora Jesus, nº2250064
- João Vasques, nº2250055
- Rui Filipe Parada, nº2211025

---

### **LLM: DistilGPT2**

## **1. Imports and Data Upload**

In [ ]:
# Instalação de bibliotecas adicionais
!pip install -q transformers datasets scikit-learn
!pip install datasets

In [ ]:
# Imports principais
from google.colab import drive
from google.colab import files
import pandas as pd
import numpy as np
import io
import torch

# Pré-processamento e métricas
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix
)

# Hugging Face - tokenização, modelo e treino
from transformers import (
    AutoTokenizer,
    GPT2ForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Conversão para dataset compatível com Hugging Face
from datasets import Dataset

In [ ]:
# Upload do dataset (em csv)

uploaded = files.upload()

review_df = pd.read_csv(io.BytesIO(uploaded['google_play_store_apps.csv']), sep=',')

Saving google_play_store_apps.csv to google_play_store_apps.csv


In [ ]:
review_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
3,10 Best Foods for You,Best idea us,Positive,1.00,0.300000
4,10 Best Foods for You,Best way,Positive,1.00,0.300000


## **2. Pré-processamento**

In [ ]:
# Remover reviews com sentimento neutro
review_df = review_df[review_df['Sentiment'] != 'Neutral']

# Visualizar as primeiras linhas
display(review_df.head(10))

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.000000,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.250000,0.288462
2,10 Best Foods for You,Works great especially going grocery store,Positive,0.400000,0.875000
3,10 Best Foods for You,Best idea us,Positive,1.000000,0.300000
4,10 Best Foods for You,Best way,Positive,1.000000,0.300000
5,10 Best Foods for You,Amazing,Positive,0.600000,0.900000
8,10 Best Foods for You,good you.,Positive,0.700000,0.600000
9,10 Best Foods for You,Useful information The amount spelling errors ...,Positive,0.200000,0.100000
10,10 Best Foods for You,"Thank you! Great app!! Add arthritis, eyes, im...",Positive,0.750000,0.875000
11,10 Best Foods for You,Greatest ever Completely awesome maintain heal...,Positive,0.992188,0.866667


## **3. Fine Tuning**

### **3.1. Codificação dos Rótulos**

In [ ]:
# Codificar sentimentos em valores numéricos
le = LabelEncoder()
review_df["label"] = le.fit_transform(review_df["Sentiment"])

# Mostrar primeiras linhas
display(review_df.head(10))

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,label
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.000000,0.533333,1
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.250000,0.288462,1
2,10 Best Foods for You,Works great especially going grocery store,Positive,0.400000,0.875000,1
3,10 Best Foods for You,Best idea us,Positive,1.000000,0.300000,1
4,10 Best Foods for You,Best way,Positive,1.000000,0.300000,1
5,10 Best Foods for You,Amazing,Positive,0.600000,0.900000,1
8,10 Best Foods for You,good you.,Positive,0.700000,0.600000,1
9,10 Best Foods for You,Useful information The amount spelling errors ...,Positive,0.200000,0.100000,1
10,10 Best Foods for You,"Thank you! Great app!! Add arthritis, eyes, im...",Positive,0.750000,0.875000,1
11,10 Best Foods for You,Greatest ever Completely awesome maintain heal...,Positive,0.992188,0.866667,1


### **3.2. Separação dos Dados em Treino e Teste**

In [ ]:
# Separar dados de treino (80%) e teste (20%)
train_df = review_df.sample(frac=0.8, random_state=42)
test_df = review_df.drop(train_df.index)

In [ ]:
# Mostrar dimensões dos conjuntos
print("train_df:", train_df.shape)
print("→", train_df.shape[0], "linhas (documentos de treino) e", train_df.shape[1], "colunas (atributos seleccionadas)")

print("\ntest_df:", test_df.shape)
print("→", test_df.shape[0], "linhas (documentos de teste) e", test_df.shape[1], "colunas (atributos seleccionadas)")

train_df: (25815, 6)
→ 25815 linhas (documentos de treino) e 6 colunas (atributos seleccionadas)

test_df: (6454, 6)
→ 6454 linhas (documentos de teste) e 6 colunas (atributos seleccionadas)


### **3.3. Conversão para Dataset Hugging Face**


In [ ]:
# Converter DataFrames para Dataset Hugging Face
train_dataset = Dataset.from_pandas(train_df[["Translated_Review", "label"]])
test_dataset = Dataset.from_pandas(test_df[["Translated_Review", "label"]])

In [ ]:
# Mostrar o conteúdo
display(train_dataset)
display(test_dataset)

Dataset({
    features: ['Translated_Review', 'label', '__index_level_0__'],
    num_rows: 25815
})

Dataset({
    features: ['Translated_Review', 'label', '__index_level_0__'],
    num_rows: 6454
})

### **3.4. Tokenização dos Conjuntos de Treino e Teste**

In [ ]:
# Carregar tokenizador do DistilGPT2
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

# Definir token de padding
tokenizer.pad_token = tokenizer.eos_token

# Tokenizar treino
train_encodings = tokenizer(
    list(train_dataset["Translated_Review"]),
    truncation=True,
    padding=True,
    max_length=512
)

# Tokenizar teste
test_encodings = tokenizer(
    list(test_dataset["Translated_Review"]),
    truncation=True,
    padding=True,
    max_length=512
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Adicionar tokens ao treino
train_dataset = train_dataset.add_column("input_ids", train_encodings["input_ids"])
train_dataset = train_dataset.add_column("attention_mask", train_encodings["attention_mask"])

# Adicionar tokens ao teste
test_dataset = test_dataset.add_column("input_ids", test_encodings["input_ids"])
test_dataset = test_dataset.add_column("attention_mask", test_encodings["attention_mask"])

### **3.5. Carregamento do Modelo e Definição dos Parâmetros de Treino**

In [ ]:
# Converter datasets para formato PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Carregar modelo para classificação binária
model = GPT2ForSequenceClassification.from_pretrained(
    "distilgpt2",
    num_labels=2
)

# Informar o modelo sobre o pad token
model.config.pad_token_id = tokenizer.pad_token_id

# Definir parâmetros de treino
training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    report_to="none"
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key                                        | Status     | 
-------------------------------------------+------------+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED | 
score.weight                               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### **3.6. Definição da Métrica de Avaliação**


In [ ]:
def calcular_metricas(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='weighted')
    rec = recall_score(labels, preds, average='weighted')
    f1 = f1_score(labels, preds, average='weighted')
    kappa = cohen_kappa_score(labels, preds)

    cm = confusion_matrix(labels, preds)

    tn, fp, fn, tp = cm.ravel()

    taxa_erro = (fp + fn) / (tp + tn + fp + fn)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "kappa": kappa,
        "taxa_erro": taxa_erro
    }

### **3.7. Treino e Avaliação do Modelo**

In [ ]:
# Configurar treino
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=calcular_metricas
)

# Treinar modelo
trainer.train()

Step,Training Loss
500,0.520167
1000,0.458586
1500,0.396575
2000,0.380799
2500,0.390117
3000,0.393662
3500,0.380713
4000,0.380089
4500,0.322493
5000,0.306549


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12908, training_loss=0.2845312165223965, metrics={'train_runtime': 3961.8008, 'train_samples_per_second': 13.032, 'train_steps_per_second': 3.258, 'total_flos': 6745619237437440.0, 'train_loss': 0.2845312165223965, 'epoch': 2.0})

In [ ]:
# Avaliar modelo
resultados = trainer.evaluate()

# Converter resultados para DataFrame
df_resultados = pd.DataFrame([{
    "Model": "DistilGPT2",
    "Acc": resultados["eval_accuracy"],
    "Prec": resultados["eval_precision"],
    "Rec": resultados["eval_recall"],
    "F1": resultados["eval_f1"],
    "Kappa": resultados["eval_kappa"],
    "Taxa_Erro": resultados["eval_taxa_erro"]
}])

# Mostrar tabela
print(df_resultados)

# Exportar CSV
df_resultados.to_csv("DistilGPT2_results.csv", index=False)

        Model       Acc      Prec       Rec        F1     Kappa  Taxa_Erro
0  DistilGPT2  0.951348  0.951874  0.951348  0.951556  0.873424   0.048652


## **4. Predição de Novo Comentário**

In [ ]:
### --- NOVOS COMENTÁRIOS: --- ###

uploaded = files.upload()

novos_comentarios = pd.read_csv(io.BytesIO(uploaded['Comentarios.csv']), sep=';')

Saving Comentarios.csv to Comentarios.csv


### **4.1. Pré-processamento**

In [ ]:
# Remover reviews com sentimento neutro
novos_comentarios = novos_comentarios[novos_comentarios['Sentiment'] != 'Neutral']

# Visualizar as primeiras linhas
display(novos_comentarios.head(10))

,Comentarios_Novos,Sentiment
0,This app is a true paragon.,Positive
1,The interface remains totally execrable.,Negative
3,Simply effulgent and highly efficient!,Positive
4,Navigation is inimical to usage.,Negative


### **4.2. Avaliação de Novos Comentários**

In [ ]:
# Extrair textos
textos = novos_comentarios["Comentarios_Novos"].astype(str).tolist()

# Tokenizar em batch
tokens = tokenizer(
    textos,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

# Garantir que o modelo está na GPU (se existir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Enviar tokens para o mesmo device
tokens = {k: v.to(device) for k, v in tokens.items()}

# Colocar modelo em modo avaliação
model.eval()

# Inferência
with torch.no_grad():
    outputs = model(**tokens)

logits = outputs.logits

# Previsões
classe_prevista = np.argmax(logits.cpu().numpy(), axis=1)

# Converter para labels originais
sentimento = le.inverse_transform(classe_prevista)

# Guardar no dataframe
novos_comentarios["Sentimento_previsto"] = sentimento

# Mostrar resultados
print(novos_comentarios[["Comentarios_Novos", "Sentiment", "Sentimento_previsto"]])

                          Comentarios_Novos Sentiment Sentimento_previsto
0               This app is a true paragon.  Positive            Positive
1  The interface remains totally execrable.  Negative            Negative
3    Simply effulgent and highly efficient!  Positive            Positive
4          Navigation is inimical to usage.  Negative            Negative
